# Kepler-64 credibility gate â€” Kaggle GPU

Runs the full harvest â†’ train â†’ match â†’ report pipeline on a free T4.
Gradient-through-physics parallelizes well on GPU; expect roughly an order
of magnitude per-step speedup vs a laptop CPU.

**If the session dies:** rerun cells 2â€“3, then resume training with
`--only train --resume kepler64/training/gate_ckpt.npz` (checkpoints are
written to the working dir every `--ckpt-every` steps). Save
`gate_ckpt.npz`, `gate_examples.pkl`, and `docs/credibility_gate_results.*`
as Kaggle Outputs before leaving.

In [ ]:
# Cell 1 — deps + accelerator check (GPU engages automatically;
# the repo's XLA device-count tuning disables itself when nvidia-smi exists)
%pip install -q optax python-chess
import os
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.20'
import jax
devices = jax.devices()
print('JAX Devices:', devices)
assert any(d.platform == 'gpu' for d in devices), 'GPU not detected! Turn on GPU in Kaggle: Settings -> Accelerator -> GPU T4'


In [ ]:
# Cell 2 â€” repo (public clone; if private, upload as a Kaggle dataset instead)
!git clone -q https://github.com/r-baruah/kepler-64.git
%cd kepler-64
!python -c "import kepler64; print('import ok')"

### Batch Size Scaling & Multi-Worker Acceleration on Kaggle T4
- **Parallel Match Execution:** `--workers 4` runs head-to-head match games concurrently across all 4 vCPUs,
  slashing the 200-game match phase from ~25 minutes down to ~6 minutes.
- **Tensor Core Mini-batches:** `--batch-size 256` keeps GPU tensor cores saturated during training.
- **Preemption resilience:** use `--append` to never lose harvested games on disconnect,
  and `--resume kepler64/training/gate_ckpt.npz` to continue training with full Adam momentum & RNG state.


In [ ]:
# Cell 3 — GPU smoke run (harvest 6 games + 10 training steps + 2 match games)
# Proves the end-to-end GPU path and multi-worker execution before a full run.
!python scripts/credibility_gate.py --games 6 --max-plies 30 --steps 10 \
  --match-games 2 --match-move-ms 50 --seed 0 --batch-size 256 --workers 4 \
  --log-every 2 --ckpt-every 5 --allow-skew


In [ ]:
# Cell 4 — full scaled credibility gate (natural decisions at max-plies 96,
# ~800 steps for a few-hundred-example harvest, skew guard on by default).
# Killed session? Rerun with: --append (harvest) + --resume kepler64/training/gate_ckpt.npz
!python scripts/credibility_gate.py --games 40 --max-plies 96 --steps 800 \
  --match-games 200 --match-move-ms 150 --seed 7 --batch-size 256 --workers 4 --append \
  --log-every 50 --ckpt-every 100 2>&1 | tee gate.log


In [ ]:
# Cell 5 — collect the claim (the only numbers allowed in public copy)
!tail -25 docs/credibility_gate_results.md
!ls -la docs/credibility_gate_results.* kepler64/training/gate_ckpt.npz kepler64/training/trained_constants_gate.json
